# 02 — Preprocesamiento
Unión de tablas CRM, limpieza, feature engineering, encoding y split train/test.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from paths import CSV_FILES, PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

accounts = pd.read_csv(CSV_FILES["accounts"])
products = pd.read_csv(CSV_FILES["products"])
sales_teams = pd.read_csv(CSV_FILES["sales_teams"])
pipeline = pd.read_csv(CSV_FILES["sales_pipeline"])

In [ ]:
df = pipeline.merge(accounts, on="account", how="left")
df = df.merge(products, on="product", how="left")
df = df.merge(sales_teams, on="sales_agent", how="left")

# Solo oportunidades cerradas (Won / Lost)
df = df[df["deal_stage"].isin(["Won", "Lost"])].copy()
df = df.dropna(subset=["account", "engage_date"])

df["engage_date"] = pd.to_datetime(df["engage_date"])
df["won"] = (df["deal_stage"] == "Won").astype(int)

print(f"Registros tras limpieza: {len(df)}")

In [ ]:
df["subsidiary_of"] = df["subsidiary_of"].fillna("None")
df["has_subsidiary"] = (df["subsidiary_of"] != "None").astype(int)

for col in ["revenue", "employees"]:
    df[col] = df.groupby("sector")[col].transform(lambda x: x.fillna(x.median()))
    df[col] = df[col].fillna(df[col].median())

df["company_age"] = df["engage_date"].dt.year - df["year_established"]
df["revenue_per_employee"] = df["revenue"] / df["employees"].replace(0, np.nan)
df["revenue_per_employee"] = df["revenue_per_employee"].fillna(df["revenue_per_employee"].median())
df["engage_year"] = df["engage_date"].dt.year
df["engage_month"] = df["engage_date"].dt.month
df["engage_quarter"] = df["engage_date"].dt.quarter
df["engage_dayofweek"] = df["engage_date"].dt.dayofweek
df["is_premium_product"] = (df["sales_price"] > df["sales_price"].median()).astype(int)

In [ ]:
cat_cols = ["sector", "office_location", "subsidiary_of", "product", "series",
            "manager", "regional_office", "sales_agent"]
num_cols = ["year_established", "revenue", "employees", "sales_price",
            "company_age", "revenue_per_employee", "engage_year", "engage_month",
            "engage_quarter", "engage_dayofweek", "has_subsidiary", "is_premium_product"]

for col in cat_cols:
    df[col] = df[col].astype(str)

X = pd.get_dummies(df[cat_cols + num_cols], columns=cat_cols, drop_first=True)
y = df["won"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Features: {X_train.shape[1]} | Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test.csv", index=False)
X_train_scaled.to_csv(PROCESSED_DIR / "X_train_scaled.csv", index=False)
X_test_scaled.to_csv(PROCESSED_DIR / "X_test_scaled.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)
joblib.dump(scaler, PROCESSED_DIR / "scaler.joblib")

print("Datos procesados guardados en:", PROCESSED_DIR)